In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from scipy.integrate import solve_ivp

np.random.seed(13)

In [ ]:
def solve_markov_task(alpha, beta, t_max, start_from_stationary=False):
    # 1. Calculate theoretical stationary distribution pi
    # Equations: pi_1 * alpha = pi_2 * beta and pi_1 + pi_2 = 1
    pi_1 = beta / (alpha + beta)
    pi_2 = alpha / (alpha + beta)
    pi_theoretical = np.array([pi_1, pi_2])

    # 2. Simulation of the trajectory (Gillespie Algorithm)
    times = [0]
    states = []
    
    # Initial state
    if start_from_stationary:
        current_state = np.random.choice([1, 2], p=pi_theoretical)
    else:
        current_state = 1 # Default start at state 1
    
    states.append(current_state)
    t = 0
    
    while t < t_max:
        # Intensity of leaving current state
        lambda_val = alpha if current_state == 1 else beta
        
        # Time spent in current state is exponentially distributed
        wait_time = np.random.exponential(1/lambda_val)
        t += wait_time
        
        if t > t_max:
            break
            
        # Switch state (since there are only 2 states, it must flip)
        current_state = 2 if current_state == 1 else 1
        
        times.append(t)
        states.append(current_state)

    # Add final point for plotting consistency
    times.append(t_max)
    states.append(states[-1])

    # 3. Estimate stationary distribution using Strong Law of Large Numbers
    # We calculate the total time spent in each state
    duration_state_1 = 0
    duration_state_2 = 0
    
    for i in range(len(times) - 1):
        duration = times[i+1] - times[i]
        if states[i] == 1:
            duration_state_1 += duration
        else:
            duration_state_2 += duration
            
    pi_estimated = np.array([duration_state_1 / t_max, duration_state_2 / t_max])
    
    return np.array(times), np.array(states), pi_theoretical, pi_estimated

# Parameters for comparison
params = [(0.1, 0.1), (0.9, 0.9)]
t_sim = 100

plt.figure(figsize=(12, 8))

for i, (a, b) in enumerate(params):
    times, states, pi_th, pi_est = solve_markov_task(a, b, t_sim)
    
    # Plotting
    plt.subplot(2, 1, i+1)
    plt.step(times, states, where='post', label=f'Trajectory (α={a}, β={b})')
    plt.ylim(0.5, 2.5)
    plt.yticks([1, 2])
    plt.title(f"Markov Process Trajectory: α={a}, β={b}")
    plt.xlabel("Time (s)")
    plt.ylabel("State")
    plt.grid(True, linestyle='--', alpha=0.7)
    
    print(f"--- Results for α={a}, β={b} ---")
    print(f"Theoretical pi: {pi_th}")
    print(f"Estimated pi:   {pi_est}")
    print("-" * 30)

plt.tight_layout()
plt.show()

# Part 4: Trajectory starting from stationary distribution
times_st, states_st, pi_th_st, _ = solve_markov_task(0.5, 0.5, 50, start_from_stationary=True)
plt.figure(figsize=(12, 4))
plt.step(times_st, states_st, where='post', color='green')
plt.title("Trajectory starting from Stationary Distribution X(0) ~ π (α=0.5, β=0.5)")
plt.yticks([1, 2])
plt.xlabel("Time (s)")
plt.ylabel("State")
plt.show()

In [ ]:

def simulate_box_process(a_star, a_dagger, t_max, initial_x):
    times = [0]
    states = [initial_x]
    t = 0
    current_x = initial_x
    
    while t < t_max:
        # Total intensity: rate of entry + rate of exit
        # Exit rate depends on current number of particles
        lambda_total = a_star + (current_x * a_dagger)
        
        if lambda_total == 0: # Should not happen with a_star > 0
            break
            
        # Time until next event
        wait_time = np.random.exponential(1 / lambda_total)
        t += wait_time
        
        if t > t_max:
            break
            
        # Determine event type: birth or death?
        # Probability of birth: a_star / lambda_total
        if np.random.rand() < (a_star / lambda_total):
            current_x += 1
        else:
            current_x -= 1
            
        times.append(t)
        states.append(current_x)
        
    return np.array(times), np.array(states)

# Parameters
a_star = 10
a_dagger = 2
t_sim = 200  # Long simulation for stationary analysis
initial_states = [0, 10, 20]

# 1. Multiple trajectories plot
plt.figure(figsize=(12, 5))
for start_x in initial_states:
    t_plot, x_plot = simulate_box_process(a_star, a_dagger, 30, start_x)
    plt.step(t_plot, x_plot, where='post', label=f'Start X(0)={start_x}')

plt.axhline(y=a_star/a_dagger, color='r', linestyle='--', label='Theoretical E[X]')
plt.title("Trajectories of the Particle Process (Birth-Death)")
plt.xlabel("Time")
plt.ylabel("Number of particles X(t)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 2. Stationary distribution estimation
# Using one long trajectory
t_long, x_long = simulate_box_process(a_star, a_dagger, 1000, 0)

# Calculate time-weighted distribution
durations = np.diff(t_long)
state_times = {}

for i in range(len(durations)):
    state = x_long[i]
    state_times[state] = state_times.get(state, 0) + durations[i]

# Normalize to get probabilities
total_time = np.sum(durations)
states_sorted = sorted(state_times.keys())
probs = [state_times[s] / total_time for s in states_sorted]

# 3. Plot Stationary Distribution (Barplot)
plt.figure(figsize=(10, 5))
plt.bar(states_sorted, probs, alpha=0.7, label='Estimated π(x)')

# Theoretical distribution: Poisson with lambda = a_star / a_dagger
from scipy.stats import poisson
lambda_stationary = a_star / a_dagger
x_vals = np.arange(0, max(states_sorted) + 1)
plt.plot(x_vals, poisson.pmf(x_vals, lambda_stationary), 'ro-', label='Theoretical Poisson(λ=5)')

plt.title("Stationary Distribution: Simulation vs Theory")
plt.xlabel("State (Number of particles)")
plt.ylabel("Probability")
plt.legend()
plt.show()

# 4. Estimating expected value
estimated_mean = sum(s * (state_times[s] / total_time) for s in state_times)
print(f"Results for a_star={a_star}, a_dagger={a_dagger}:")
print(f"Theoretical lim E[X(t)] = a_star / a_dagger = {a_star/a_dagger}")
print(f"Estimated mean from simulation = {estimated_mean:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson

def simulate_two_boxes(a_s1, a_12, a_2d, t_max, init_x1=0, init_x2=0):
    times = [0]
    states = [(init_x1, init_x2)]
    t = 0
    cur_x1, cur_x2 = init_x1, init_x2
    
    while t < t_max:
        # Intensities
        rate_in = a_s1
        rate_transfer = cur_x1 * a_12
        rate_out = cur_x2 * a_2d
        
        total_rate = rate_in + rate_transfer + rate_out
        
        if total_rate == 0: break # Safety
            
        wait_time = np.random.exponential(1 / total_rate)
        t += wait_time
        if t > t_max: break
            
        # Choose event
        r = np.random.rand() * total_rate
        if r < rate_in:
            cur_x1 += 1
        elif r < rate_in + rate_transfer:
            cur_x1 -= 1
            cur_x2 += 1
        else:
            cur_x2 -= 1
            
        times.append(t)
        states.append((cur_x1, cur_x2))
        
    return np.array(times), np.array(states)

# Parameters
a_s1, a_12, a_2d = 10, 2, 2
t_sim = 2000 # Long time for stationary distribution

times, states = simulate_two_boxes(a_s1, a_12, a_2d, t_sim)
x1_traj, x2_traj = states[:, 0], states[:, 1]

# 1. Visualization of trajectories
plt.figure(figsize=(12, 5))
plt.step(times[:500], x1_traj[:500], label='Box 1 ($X_1$)', alpha=0.8)
plt.step(times[:500], x2_traj[:500], label='Box 2 ($X_2$)', alpha=0.8)
plt.title("Trajectories of the 2-Box System")
plt.xlabel("Time")
plt.ylabel("Number of particles")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 2. Estimation of Stationary Distribution and Covariance
durations = np.diff(times)
# Covariance: E[X1*X2] - E[X1]*E[X2] using time-weighted averages
mean_x1 = np.sum(x1_traj[:-1] * durations) / times[-1]
mean_x2 = np.sum(x2_traj[:-1] * durations) / times[-1]
mean_x1x2 = np.sum(x1_traj[:-1] * x2_traj[:-1] * durations) / times[-1]
cov_12 = mean_x1x2 - (mean_x1 * mean_x2)

print(f"--- Simulation Results (t={t_sim}) ---")
print(f"Mean X1: {mean_x1:.3f} (Theory: {a_s1/a_12:.1f})")
print(f"Mean X2: {mean_x2:.3f} (Theory: {a_s1/a_2d:.1f})")
print(f"Estimated Cov(X1, X2): {cov_12:.5f}")

# 3. 2D Histogram (Joint Distribution)
plt.figure(figsize=(8, 6))
plt.hist2d(x1_traj, x2_traj, bins=[np.arange(15), np.arange(15)], density=True, cmap='Blues')
plt.colorbar(label='Probability')
plt.title("Estimated Joint Stationary Distribution $\pi(x_1, x_2)$")
plt.xlabel("Particles in Box 1")
plt.ylabel("Particles in Box 2")
plt.show()

### 2.14

#### Task
We analyze *Markov Process* $X(t)$ which describe amount of particles in *left side* of the box, with all particles equal to $r$. 

##### Definitions and assumptions 

* $X(t) \in \{0, 1, \dots, r\}$ - state of system
* Intensity matrix is defined as follows:
  * $q(x, x-1) = ax$
  * $q(x, x+1) = b(r-x)$
  * $q(x, x) = -(ax + b(r-x))$.

#### Solution

##### Stationary distribution 

We need to solve following equation:
$$\pi(x) q(x, x+1) = \pi(x+1) q(x+1, x)$$
We use definition of intensities
$$\pi(x) \cdot b(r-x) = \pi(x+1) \cdot a(x+1)$$
We rearrange equations to obtain recurrence formula 
$$\pi(x+1) = \frac{b(r-x)}{a(x+1)} \pi(x)$$
From which we obtain following equation
$$\pi(x) = \pi(0) \cdot \frac{b^x}{a^x} \cdot \frac{r(r-1)\dots(r-x+1)}{x!} = \pi(0) \binom{r}{x} \left(\frac{b}{a}\right)^x$$
From normalization condition we obtain
$$\begin{align*}
1 &= \pi(0) \sum_{x=0}^r \binom{r}{x}  \left(\frac{b}{a}\right)^x = \pi(0) \left(1 + \frac{b}{a}\right)^r \\
\pi(0) &= \left(\frac{a}{a+b}\right)^r
\end{align*}$$
As a result we obtain binomial distribution:
$$\pi(x) = \binom{r}{x} \left(\frac{b}{a+b}\right)^x \left(\frac{a}{a+b}\right)^{r-x}$$
so $X \sim Bin(r, p)$ where $p = \frac{b}{a+b}$.

##### Relation with 
Ehrenfest model is sum of $r$ independent proces with following $Q_i$ matrix:
$$Q_i= \begin{pmatrix}-a & a \\ b & -b\end{pmatrix}$$
For which stationary distribution is:
$$\begin{align*}
q_1 &= \frac{b}{a+b} \\
q_2 &= \frac{a}{a+b} 
\end{align*}
$$
THat is 
$$X = \sum_{i=1}^r X_i,$$
where $X_i$ is described by $Q_i$  

In [ ]:
# =============================================================================
# EXERCISE 2.14: EHRENFEST MODEL (CONTINUOUS TIME)
# =============================================================================

def simulate_ehrenfest(r, a, b, t_max, x0):
    """
    Simulates the Ehrenfest model using the Gillespie algorithm.
    
    Parameters:
    r: int - Total number of particles
    a: float - Rate of moving Left -> Right
    b: float - Rate of moving Right -> Left
    t_max: float - Simulation time horizon
    x0: int - Initial number of particles in the left part
    """
    times = [0.0]
    states = [x0]
    t = 0.0
    x = x0
    
    while t < t_max:
        # Rates for transitions
        rate_out = x * a         # L -> R
        rate_in = (r - x) * b    # R -> L
        total_rate = rate_out + rate_in
        
        if total_rate == 0:
            break
            
        # Time until the next event
        dt = np.random.exponential(1 / total_rate)
        t += dt
        
        # Determine which event occurs
        if np.random.rand() < rate_out / total_rate:
            x -= 1
        else:
            x += 1
            
        times.append(t)
        states.append(x)
        
    return np.array(times), np.array(states)

def plot_ehrenfest_results(r, a, b, t_max, x0, t_long_horizon=5000):
    """
    Plots the trajectory and the stationary distribution comparison.
    """
    print(f"--- Ehrenfest Model Parameters ---")
    print(f"Particles (r): {r}, a: {a}, b: {b}, t_max: {t_max}, x0: {x0}\n")

    # 1. Trajectory Plot
    t, x = simulate_ehrenfest(r, a, b, t_max, x0)
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.step(t, x, where='post', label='Trajectory $X(t)$')
    plt.axhline(r * b / (a + b), color='red', linestyle='--', label='Theoretical Mean')
    plt.xlabel('Time $t$')
    plt.ylabel('Particles in Left Part')
    plt.title('Ehrenfest Model Trajectory')
    plt.legend()
    plt.grid(True)

    # 2. Stationary Distribution Analysis (Long Simulation)
    t_long, x_long = simulate_ehrenfest(r, a, b, t_long_horizon, x0)
    durations = np.diff(t_long)
    counts = {}
    for i in range(len(durations)):
        state = x_long[i]
        counts[state] = counts.get(state, 0) + durations[i]

    sim_probs = {k: v / t_long[-1] for k, v in counts.items()}
    
    # Theoretical Binomial(r, p) where p = b / (a + b)
    p_theory = b / (a + b)
    x_vals = np.arange(0, r + 1)
    theory_probs = binom.pmf(x_vals, r, p_theory)

    plt.subplot(1, 2, 2)
    plt.bar(sim_probs.keys(), sim_probs.values(), alpha=0.5, label='Simulation (Time-weighted)')
    plt.plot(x_vals, theory_probs, 'ro-', markersize=2, label=f'Theoretical $Bin(r, {p_theory:.2f})$')
    plt.xlabel('State $x$')
    plt.ylabel('Probability')
    plt.title('Stationary Distribution')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_ehrenfest_results(r=100, a=1.0, b=1.5, t_max=50, x0=0)

### 2.15

#### Task 

We consider *birth-death* process for seek amount of *infected* at time $t$: $I(t)$. We consider deterministic and stochastic approach:

##### Deterministic approach
For *deterministic* we consider following differential equation:
$$\frac{dI(t)}{dt} = (\alpha - \beta)I(t)$$
With initial condition $I(0) = i_0$ we obtain:
$$I(t) = i_0 e^{(\alpha - \beta)t}$$

> This model describe macroscopic effect

##### Stochastic approach
For *stochastic* approach we consider following probabilities:
* $q(i, i+1) = \alpha i$.
* $q(i, i-1) = \beta i$.

In [ ]:
# =============================================================================
# EXERCISE 2.15: EARLY EPIDEMIC PHASE
# =============================================================================

def simulate_epidemic_early(alpha, beta, i0, t_max):
    """
    Simulates the early phase of an epidemic (linear birth-death process).
    
    Parameters:
    alpha: float - Infection rate
    beta: float - Recovery/death rate
    i0: int - Initial infected count
    """
    times = [0.0]
    states = [i0]
    t, i = 0.0, i0
    
    while t < t_max and i > 0:
        rate_inf = alpha * i
        rate_rec = beta * i
        total_rate = rate_inf + rate_rec
        
        dt = np.random.exponential(1 / total_rate)
        t += dt
        
        if np.random.rand() < rate_inf / total_rate:
            i += 1
        else:
            i -= 1
            
        times.append(t)
        states.append(i)
        if i > 1e5: # Safety break for memory
            break
    return np.array(times), np.array(states)

def plot_epidemic_early_results(alpha, beta, i0, t_max, n_trajectories=10):
    """
    Plots stochastic trajectories against the deterministic ODE solution.
    """
    print(f"--- Early Epidemic Parameters ---")
    print(f"Alpha: {alpha}, Beta: {beta}, i0: {i0}, t_max: {t_max}\n")

    plt.figure(figsize=(10, 6))
    
    # ODE solution: I(t) = i0 * exp((alpha - beta) * t)
    t_eval = np.linspace(0, t_max, 200)
    i_ode = i0 * np.exp((alpha - beta) * t_eval)
    
    for _ in range(n_trajectories):
        t_sim, i_sim = simulate_epidemic_early(alpha, beta, i0, t_max)
        plt.step(t_sim, i_sim, where='post', alpha=0.5)

    plt.plot(t_eval, i_ode, 'k--', linewidth=2, label='ODE: $dI/dt = (\\alpha-\\beta)I$')
    plt.yscale('log')
    plt.xlabel('Time $t$')
    plt.ylabel('Infected $I(t)$ (Log Scale)')
    plt.title('Early Epidemic Phase: Stochastic vs Deterministic')
    plt.legend()
    plt.grid(True, which="both", ls="-")
    plt.show()

In [ ]:
plot_epidemic_early_results(alpha=2.0, beta=1.5, i0=1, t_max=10)

### 2.16

### 2.16

#### Task
We analyze the *SIR (Susceptible-Infectious-Removed) epidemic model* in a population of size $\ell$. We aim to describe the transition from a microscopic stochastic Markov process to a macroscopic deterministic system of Differential Equations.

##### Definitions and assumptions

* **State of the system:** $(S(t), I(t)) \in \{0, 1, \dots, \ell\}^2$, where:
    * $S(t)$ is the number of Susceptible individuals.
    * $I(t)$ is the number of Infectious individuals.
    * $R(t) = \ell - S(t) - I(t)$ is the number of Removed individuals.
* **Intensity matrix (Transitions):**
    * Infection: $q((s, i), (s-1, i+1)) = \alpha \frac{s}{\ell} i$
    * Recovery: $q((s, i), (s, i-1)) = \beta i$
    * Total exit rate: $q((s, i), (s, i)) = -(\alpha \frac{s}{\ell} i + \beta i)$



#### Solution

##### Kurtz Theorem
To justify the relation between the stochastic simulation and the ODE, i use **Kurtz’s Theorem** (also known as the Law of Large Numbers for density dependent Markov processes).

* **Assumptions:**
    1.  Let $X_\ell(t) = \frac{1}{\ell}(S(t), I(t))$ be the density process.
    2.  Transitions can be written as $x \to x + \frac{1}{\ell} \nu_j$ with intensity $\ell \lambda_j(x)$, where $\nu_j$ are jump vectors.
    3.  $\lambda_j(x)$ are Lipschitz continuous.
* **Thesis:**
    As the population size $\ell \to \infty$, the stochastic process $X_\ell(t)$ converges in probability to the deterministic trajectory $x(t)$:
    $$\lim_{\ell \to \infty} P\left( \sup_{0 \le u \le t} |X_\ell(u) - x(u)| > \epsilon \right) = 0$$
    where $x(t)$ is the solution to the system of ODEs:
    $$\frac{dx}{dt} = \sum_{j} \nu_j \lambda_j(x) = F(x)$$

##### Macroscopic ODE Derivation
Using the jump vectors and intensities from our definitions:
1.  **Infection jump:** $\nu_{inf} = (-1, 1)$ with intensity $\lambda_{inf} = \alpha s i / \ell^2 \to \alpha S I$ in density.
2.  **Recovery jump:** $\nu_{rec} = (0, -1)$ with intensity $\lambda_{rec} = \beta i / \ell \to \beta I$ in density.

By applying the thesis of Kurtz's Theorem, we obtain the classical SIR differential equations:
$$\begin{cases} 
\frac{ds}{dt} = -\alpha s i \\
\frac{di}{dt} = \alpha s i - \beta i 
\end{cases}$$
where $s = S/\ell$ and $i = I/\ell$.



<!-- ##### Comparison with Task 2.15
* **Linearity:** In Task 2.15 (Early Epidemic), we assumed $S(t) \approx \ell$, leading to a constant infection rate per individual ($\alpha i$). This resulted in a linear Birth-Death process and exponential growth.
* **Non-linearity:** In Task 2.16, we account for the depletion of the susceptible population. The rate $\alpha \frac{S}{\ell} I$ introduces a quadratic interaction term, which causes the epidemic to peak and eventually decline as the "fuel" (Susceptibles) is exhausted.
* **Stability:** In both tasks, the stochastic mean $E[I(t)]$ follows the ODE, but Kurtz's Theorem proves that for large $\ell$, the relative variance of the stochastic flucutations vanishes, making the deterministic model highly accurate. -->

In [ ]:
def simulate_sir(alpha, beta, ell, i0, s0, t_max):
    """
    Simulates the SIR model using the Gillespie algorithm.
    
    Parameters:
    alpha: float - Infection rate
    beta: float - Recovery rate
    ell: int - Total population size
    i0: int - Initial infected
    s0: int - Initial susceptible
    """
    times = [0.0]
    S = [s0]
    I = [i0]
    t, s, i = 0.0, s0, i0
    
    while t < t_max and i > 0:
        rate_inf = alpha * (s / ell) * i
        rate_rec = beta * i
        total_rate = rate_inf + rate_rec
        
        dt = np.random.exponential(1 / total_rate)
        t += dt
        
        if np.random.rand() < rate_inf / total_rate:
            s -= 1
            i += 1
        else:
            i -= 1
            
        times.append(t)
        S.append(s)
        I.append(i)
        
    return np.array(times), np.array(S), np.array(I)

def plot_sir_results(alpha, beta, ell, i0, s0, t_max, n_trajectories=5):
    """
    Plots stochastic SIR trajectories against the numerical ODE solution.
    """
    print(f"--- SIR Model Parameters ---")
    print(f"Alpha: {alpha}, Beta: {beta}, Population (ell): {ell}, i0: {i0}, s0: {s0}\n")

    # Numerical ODE solution
    def sir_ode(t, y):
        s, i = y
        dsdt = -alpha * (s / ell) * i
        didt = (alpha * (s / ell) - beta) * i
        return [dsdt, didt]

    sol = solve_ivp(sir_ode, [0, t_max], [s0, i0], t_eval=np.linspace(0, t_max, 500))

    plt.figure(figsize=(10, 6))
    
    # Stochastic simulations
    for j in range(n_trajectories):
        ts, ss, isi = simulate_sir(alpha, beta, ell, i0, s0, t_max)
        label = 'Stochastic I(t)' if j == 0 else None
        plt.step(ts, isi, where='post', alpha=0.4, label=label)

    plt.plot(sol.t, sol.y[1], 'r-', linewidth=2.5, label='Deterministic ODE I(t)')
    plt.xlabel('Time $t$')
    plt.ylabel('Number of Infected Individuals')
    plt.title(f'SIR Model Simulation (N={ell})')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
plot_sir_results(alpha=2.0, beta=1.0, ell=1000, i0=1, s0=999, t_max=25)